In [4]:
model_a_features = [
    'price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
    'score_ch1', 'score_ch2', 'score_ch3', 'score_ch4', 'score_ch5'
]

In [5]:
model_b_features = {
    'sneakers': ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch3', 'score_ch4'],   # Granger SIGNIFICANT
    'cards':    ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch1'],                 # Granger SIGNIFICANT
    'lego':     ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch1'],                 # Granger MARGINAL (p=0.054)
}

In [6]:
import os
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

df = pd.read_csv('xgboost_data.csv')
# lag 변수 NaN 제거 (lag1~3 생성으로 앞 행 결측)
df = df.dropna(subset=['price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3']).copy()
df = df.sort_values(['asset_type', 'year_month']).reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)
results_1 = []

for asset in ['sneakers', 'cards', 'lego']:
    subset = df[df['asset_type'] == asset].copy().reset_index(drop=True)
    # 월(month) 기준으로 분할 — 행 번호 기준 분할 시 아이템이 섞여 시간 순서 깨짐
    months = sorted(subset['year_month'].unique())

    X_a = subset[model_a_features]
    X_b = subset[model_b_features[asset]]
    y   = subset['price_direction']

    auc_a_folds, auc_b_folds = [], []

    for train_m_idx, test_m_idx in tscv.split(months):
        train_months = set(months[i] for i in train_m_idx)
        test_months  = set(months[i] for i in test_m_idx)

        train_mask = subset['year_month'].isin(train_months)
        test_mask  = subset['year_month'].isin(test_months)

        # 테스트 폴드 단일 클래스 → AUC 계산 불가, 건너뜀
        if len(y[test_mask].unique()) < 2:
            continue

        clf_a = xgb.XGBClassifier(n_estimators=100, max_depth=3,
                                   learning_rate=0.05, random_state=42,
                                   eval_metric='auc', verbosity=0)
        clf_a.fit(X_a[train_mask], y[train_mask])
        pred_a = clf_a.predict_proba(X_a[test_mask])[:, 1]
        auc_a_folds.append(roc_auc_score(y[test_mask], pred_a))

        clf_b = xgb.XGBClassifier(n_estimators=100, max_depth=3,
                                   learning_rate=0.05, random_state=42,
                                   eval_metric='auc', verbosity=0)
        clf_b.fit(X_b[train_mask], y[train_mask])
        pred_b = clf_b.predict_proba(X_b[test_mask])[:, 1]
        auc_b_folds.append(roc_auc_score(y[test_mask], pred_b))

    results_1.append({
        'asset':        asset,
        'auc_a_mean':   np.mean(auc_a_folds),
        'auc_b_mean':   np.mean(auc_b_folds),
        'auc_a_folds':  auc_a_folds,
        'auc_b_folds':  auc_b_folds,
    })
    print(f"{asset}: Model A={np.mean(auc_a_folds):.4f}  Model B={np.mean(auc_b_folds):.4f}  ({len(auc_a_folds)} folds)")

os.makedirs('results', exist_ok=True)
pd.DataFrame(results_1)[['asset', 'auc_a_mean', 'auc_b_mean']].to_csv(
    'results/xgboost_auc.csv', index=False)
print("Saved → results/xgboost_auc.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'xgboost_data.csv'

In [ ]:
import matplotlib.pyplot as plt
import os

res_df = pd.DataFrame(results_1).set_index('asset')
plot_data = res_df[['auc_a_mean', 'auc_b_mean']]

ax = plot_data.plot(kind='bar', figsize=(10, 6), color=['#5A9BD4', '#F15A60'], width=0.6)

plt.title('XGBoost Model AUC Comparison by Asset Type (Higher is Better)',
          fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Asset Type', fontsize=12)
plt.ylabel('Mean AUC (TimeSeriesSplit, 5-fold)', fontsize=12)
plt.xticks(rotation=0, fontsize=11)
plt.ylim(0, 1.1)
plt.legend(['Model A (All Channels)', 'Model B (Granger-selected Channels)'], fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.3f}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, xytext=(0, 4),
                    textcoords='offset points')

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/xgboost_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()